<a href="https://colab.research.google.com/github/soominlee0726/Email_Generator_for_students/blob/main/Email_Generator_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Final Model of email generator

In [ ]:
!pip -q install "transformers>=4.37.0" accelerate sentencepiece

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
def generate_email_fewshot(user_input):
    system_prompt = """
너는 대학생을 도와 교수님께 보내는 정중한 한국어 이메일을 작성하는 assistant다.
항상 이메일 본문만 출력하고, 공손하고 자연스러운 메일 문체를 사용하라.
학생 이름은 항상 OOO으로 작성하라. 메일 구조는 인사, 자기소개, 상황 설명, 요청 또는 문의, 마무리 흐름으로 작성하라.
"""

    example_user_1 = "박정우 교수님께 회로이론 수업 과제 제출 연장 관련 메일 작성. 개인적인 사정으로 과제를 기한 내 제출하기 어려워 하루 정도 연장이 가능한지 여쭙고 싶습니다."
    example_assistant_1 = """안녕하세요, 박정우 교수님.
저는 교수님의 회로이론 수업을 수강 중인 OOO입니다.

개인적인 사정으로 인해 이번 과제를 기한 내에 제출하기 어려운 상황이 되어 메일 드리게 되었습니다. 혹시 가능하시다면 제출 기한을 하루 정도 연장해주실 수 있을지 조심스럽게 여쭙고 싶습니다. 갑작스럽게 부탁드리게 되어 죄송합니다.

검토해주시면 감사하겠습니다.
OOO 드림"""

    example_user_2 = "최민지 교수님께 자료구조 수업 면담 요청 관련 메일 작성. 프로젝트 방향에 대해 조언을 받고 싶어 면담을 부탁드리고 싶습니다."
    example_assistant_2 = """안녕하세요, 최민지 교수님.
저는 교수님의 자료구조 수업을 수강 중인 OOO입니다.

현재 진행 중인 프로젝트의 방향과 관련하여 교수님께 조언을 구하고 싶어 메일 드립니다. 가능하시다면 교수님께서 편하신 시간에 잠시 면담을 요청드리고 싶습니다. 짧은 시간이라도 괜찮으니 가능 여부를 알려주시면 감사하겠습니다.

감사합니다.
OOO 드림"""

    example_user_3 = "김준호 교수님께 자연어처리 수업 출결 관련 메일 작성해줘. 개인 사정으로 인해 이번 주 금요일 수업에 참석하지 못할 것 같습니다."
    example_assistant_3 = """안녕하세요, 김준호 교수님.
저는 교수님의 자연어처리 수업을 수강 중인 OOO입니다.

개인적인 사정으로 인해 부득이하게 수업에 참석하지 못할 것 같아 미리 말씀드리고자 메일 드립니다. 수업에 불참하게 되어 죄송합니다. 혹시 결석과 관련하여 별도로 안내해주실 사항이 있다면 확인 부탁드립니다.

감사합니다.
OOO 드림"""

    example_user_4 = "김하늘 교수님께 컴퓨터비전 수업 연구실 관련 문의 메일 작성. 교수님 연구실 학부연구생 모집 여부를 여쭙고 싶습니다."
    example_assistant_4 = """안녕하세요, 김하늘 교수님.
저는 교수님의 컴퓨터비전 수업을 수강 중인 OOO입니다.

수업을 들으며 교수님의 연구 분야에 큰 관심을 가지게 되어 메일 드립니다. 혹시 현재 교수님 연구실에서 학부연구생을 모집하고 계신지 여쭙고 싶습니다. 가능하시다면 지원 방법이나 필요한 준비 사항에 대해서도 안내해주시면 감사하겠습니다.

읽어주셔서 감사합니다.
OOO 드림"""


    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": example_user_1},
        {"role": "assistant", "content": example_assistant_1},
        {"role": "user", "content": example_user_2},
        {"role": "assistant", "content": example_assistant_2},
        {"role": "user", "content": user_input},
        {"role": "user", "content": example_user_3},
        {"role": "assistant", "content": example_assistant_3},
        {"role": "user", "content": user_input},
        {"role": "user", "content": example_user_4},
        {"role": "assistant", "content": example_assistant_4},
        {"role": "user", "content": user_input},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=300,
        do_sample=False,
        num_beams=4
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [ ]:
test_input = "김땡땡 교수님께 컴퓨터 수학 수업 과제 연장 문의 관련 메일 작성. 과제 기한이 촉박하여 과제 기한 연장을 문의드립니다."
print(generate_email_fewshot(test_input))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


안녕하세요, 김땡땡 교수님.
저는 교수님의 컴퓨터 수학 수업을 수강 중인 OOO입니다.

개인적인 사정으로 인해 이번 과제를 기한 내에 제출하기 어려운 상황이 되어 메일 드리게 되었습니다. 혹시 가능하시다면 제출 기한을 하루 정도 연장해주실 수 있을지 조심스럽게 여쭙고 싶습니다. 갑작스럽게 부탁드리게 되어 죄송합니다.

검토해주시면 감사하겠습니다.
OOO 드림
